# Loan default

Compare four logistic-regression models with gradient boosting for twelve-month
loan default. Inputs include loan size, income, term, interest rate, projected
return, country, and the lender's risk rating.

The notebook reads `credit_data.xlsx`. If missing, [prepare_data.py](prepare_data.py)
downloads and prepares it. Both loan workbooks stay local and are excluded from Git.

## 1. Load data

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

MODEL_DATA_FILE = Path("credit_data.xlsx")
if not MODEL_DATA_FILE.exists():
    from prepare_data import prepare_data

    prepare_data()

loans = pd.read_excel(MODEL_DATA_FILE, sheet_name="loan_data")
loans.shape

(212978, 10)

## 2. Outcome and loan order

In [2]:
loans = loans.rename(columns={"has_default_within_12_months": "target_default_12m"})
loans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 212978 entries, 0 to 212977
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   loan_id                212978 non-null  object        
 1   country                212978 non-null  object        
 2   loan_issued_at         212978 non-null  datetime64[ns]
 3   issued_amount          212978 non-null  float64       
 4   initial_interest_rate  212978 non-null  float64       
 5   initial_loan_duration  212978 non-null  int64         
 6   combined_income        212978 non-null  float64       
 7   target_default_12m     212978 non-null  int64         
 8   projected_npv_return   212978 non-null  float64       
 9   customer_risk_rating   212978 non-null  object        
dtypes: datetime64[ns](1), float64(4), int64(2), object(3)
memory usage: 16.2+ MB


In [3]:
loans = loans.sort_values(by=["loan_issued_at", "loan_id"])
loans = loans.reset_index(drop=True)
# Denmark is held out for the country check.
loans["loan_group"] = np.where(loans["country"] == "Denmark", "Denmark", "main")

## 3. Train, validation and test sets

- Training: October 2023 to June 2024.
- Validation: July to September 2024.
- Test: October 2024 to March 2025.

Denmark is held out for a separate country check.

In [4]:
ts1 = pd.Timestamp("2024-07-01")
ts2 = pd.Timestamp("2024-10-01")
loans["split"] = np.select(
    [loans["loan_issued_at"] < ts1, loans["loan_issued_at"] < ts2],
    ["train", "validation"],
    default="test",
)

In [5]:
main_loans = loans[loans["loan_group"] == "main"].copy()
denmark_loans = loans[loans["loan_group"] == "Denmark"].copy()
train = main_loans[main_loans["split"] == "train"].copy()
validation = main_loans[main_loans["split"] == "validation"].copy()
test = main_loans[main_loans["split"] == "test"].copy()

len(train), len(validation), len(test), len(denmark_loans)

(78252, 48657, 85887, 182)

## 4. Model inputs

In [6]:
# Use training statistics for all datasets.
selected_coloumns = ["issued_amount", "combined_income", "initial_loan_duration"]
col_1 = train[selected_coloumns]
mean = col_1.mean()
std = col_1.std()
train_std = (col_1 - mean) / std
val_std = (validation[selected_coloumns] - mean) / std
test_std = (test[selected_coloumns] - mean) / std
denmark_std = (denmark_loans[selected_coloumns] - mean) / std

In [7]:
for i in train["country"].unique():
    train_std[f"{i}"] = np.select([train["country"] == f"{i}"], [1], default=0)
    val_std[f"{i}"] = np.select([validation["country"] == f"{i}"], [1], default=0)
    test_std[f"{i}"] = np.select([test["country"] == f"{i}"], [1], default=0)
    denmark_std[f"{i}"] = np.select(
        [denmark_loans["country"] == f"{i}"], [1], default=0
    )

## 5. Model 1: basic loan features

In [8]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
fit = model.fit(train_std, train["target_default_12m"])
fit.coef_, fit.intercept_

(array([[ 0.11094229, -0.21948194,  0.25935695, -0.2082163 , -0.87729786,
         -0.27998932,  1.36549136]]),
 array([-2.18164823]))

### 5.1. Validation results

In [9]:
# Column 1 gives the default probability.
probabilities = fit.predict_proba(val_std)[:, 1]
# Predicted %, observed %, difference in percentage points.
(
    probabilities.mean() * 100,
    validation["target_default_12m"].mean() * 100,
    probabilities.mean() * 100 - validation["target_default_12m"].mean() * 100,
)

(8.415603369560383, 8.695562817271924, -0.2799594477115406)

In [10]:
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

auc = roc_auc_score(validation["target_default_12m"], probabilities)
brier = brier_score_loss(validation["target_default_12m"], probabilities)
# Baseline: the training default rate for every loan.
brier_baseline = (
    (validation["target_default_12m"] - train["target_default_12m"].mean())
    .pow(2)
    .mean()
)

auc, brier, brier_baseline

(0.6257099260598726, 0.07764830620256985, 0.07939855124713383)

In [11]:
# Five groups with similar loan counts, ordered by predicted risk.
observed, predicted = calibration_curve(
    validation["target_default_12m"],
    probabilities,
    n_bins=5,
    strategy="quantile",
)
pd.DataFrame(
    {
        "predicted_%": predicted * 100,
        "observed_%": observed * 100,
    }
)

,predicted_%,observed_%
0,4.077620,4.408138
1,5.902295,6.422773
2,8.151242,7.470201
3,9.377718,11.109969
4,14.569081,14.066995


### 5.2. Test results

In [12]:
test_probabilities = fit.predict_proba(test_std)[:, 1]
test_auc = roc_auc_score(test["target_default_12m"], test_probabilities)
test_brier = brier_score_loss(test["target_default_12m"], test_probabilities)
test_brier_baseline = (
    (test["target_default_12m"] - train["target_default_12m"].mean()).pow(2).mean()
)

test_auc, test_brier, test_brier_baseline

(0.6158314043368551, 0.06893434720065941, 0.06973868779197433)

In [13]:
test_observed, test_predicted = calibration_curve(
    test["target_default_12m"],
    test_probabilities,
    n_bins=5,
    strategy="quantile",
)

pd.DataFrame(
    {
        "predicted_%": test_predicted * 100,
        "observed_%": test_observed * 100,
    }
)

,predicted_%,observed_%
0,4.274602,3.329840
1,6.349621,5.827560
2,8.933172,8.505560
3,11.090954,8.814112
4,17.817819,11.182908


In [14]:
# Predicted %, observed %, AUC, Brier, baseline Brier.
(
    test_probabilities.mean() * 100,
    test["target_default_12m"].mean() * 100,
    test_auc,
    test_brier,
    test_brier_baseline,
)

(9.693265163834013,
 7.531989707406243,
 0.6158314043368551,
 0.06893434720065941,
 0.06973868779197433)

### 5.3. Denmark results

Denmark's 182 loans use the training scale, with all country indicators set to zero.

In [15]:
denmark_probabilities = fit.predict_proba(denmark_std)[:, 1]
denmark_target = denmark_loans["target_default_12m"]

pd.Series(
    {
        "Loans": len(denmark_target),
        "Actual defaults": denmark_target.sum(),
        "Predicted default (%)": denmark_probabilities.mean() * 100,
        "Observed default (%)": denmark_target.mean() * 100,
        "Model Brier": brier_score_loss(denmark_target, denmark_probabilities),
        "Baseline Brier": (denmark_target - train["target_default_12m"].mean())
        .pow(2)
        .mean(),
    }
)

Loans                    182.000000
Actual defaults           72.000000
Predicted default (%)      8.458219
Observed default (%)      39.560440
Model Brier                0.331602
Baseline Brier             0.335636
dtype: float64

## 6. Model 2: add interest rate

In [16]:
train_interest = train_std.copy()
val_interest = val_std.copy()

rate_mean = train["initial_interest_rate"].mean()
rate_std = train["initial_interest_rate"].std()

train_interest["initial_interest_rate"] = (
    train["initial_interest_rate"] - rate_mean
) / rate_std

val_interest["initial_interest_rate"] = (
    validation["initial_interest_rate"] - rate_mean
) / rate_std

interest_model = LogisticRegression()
interest_fit = interest_model.fit(train_interest, train["target_default_12m"])

interest_probabilities = interest_fit.predict_proba(val_interest)[:, 1]

interest_auc = roc_auc_score(validation["target_default_12m"], interest_probabilities)
interest_brier = brier_score_loss(
    validation["target_default_12m"], interest_probabilities
)

pd.DataFrame(
    {
        "AUC": [auc, interest_auc],
        "Brier": [brier, interest_brier],
    },
    index=["Original model", "With interest rate"],
)

,AUC,Brier
Original model,0.625710,0.077648
With interest rate,0.641143,0.077586


### 6.1. Validation calibration

In [17]:
interest_observed, interest_predicted = calibration_curve(
    validation["target_default_12m"],
    interest_probabilities,
    n_bins=5,
    strategy="quantile",
)

pd.DataFrame(
    {
        "predicted_%": interest_predicted * 100,
        "observed_%": interest_observed * 100,
    }
)

,predicted_%,observed_%
0,3.933034,4.336210
1,5.714448,6.196691
2,7.841649,7.244887
3,9.245646,10.296989
4,15.461363,15.402795


### 6.2. Test results

In [18]:
test_interest = test_std.copy()
test_interest["initial_interest_rate"] = (
    test["initial_interest_rate"] - rate_mean
) / rate_std

interest_test_probabilities = interest_fit.predict_proba(test_interest)[:, 1]
interest_test_auc = roc_auc_score(
    test["target_default_12m"], interest_test_probabilities
)
interest_test_brier = brier_score_loss(
    test["target_default_12m"], interest_test_probabilities
)

display(
    pd.Series(
        {
            "Predicted default (%)": interest_test_probabilities.mean() * 100,
            "Observed default (%)": test["target_default_12m"].mean() * 100,
            "AUC": interest_test_auc,
            "Model Brier": interest_test_brier,
            "Baseline Brier": test_brier_baseline,
        }
    )
)

interest_test_observed, interest_test_predicted = calibration_curve(
    test["target_default_12m"],
    interest_test_probabilities,
    n_bins=5,
    strategy="quantile",
)

pd.DataFrame(
    {
        "predicted_%": interest_test_predicted * 100,
        "observed_%": interest_test_observed * 100,
    }
)

Predicted default (%)    9.774926
Observed default (%)     7.531990
AUC                      0.635451
Model Brier              0.069404
Baseline Brier           0.069739
dtype: float64

,predicted_%,observed_%
0,3.998380,3.329840
1,6.088852,5.402573
2,8.643757,7.603190
3,10.680667,8.261047
4,19.462747,13.063220


### 6.3. Denmark results

In [19]:
denmark_interest = denmark_std.copy()
denmark_interest["initial_interest_rate"] = (
    denmark_loans["initial_interest_rate"] - rate_mean
) / rate_std

interest_denmark_probabilities = interest_fit.predict_proba(denmark_interest)[:, 1]
interest_denmark_brier = brier_score_loss(
    denmark_target, interest_denmark_probabilities
)

pd.Series(
    {
        "Loans": len(denmark_target),
        "Actual defaults": denmark_target.sum(),
        "Predicted default (%)": interest_denmark_probabilities.mean() * 100,
        "Observed default (%)": denmark_target.mean() * 100,
        "Model Brier": interest_denmark_brier,
        "Baseline Brier": (denmark_target - train["target_default_12m"].mean())
        .pow(2)
        .mean(),
    }
)

Loans                    182.000000
Actual defaults           72.000000
Predicted default (%)      8.413737
Observed default (%)      39.560440
Model Brier                0.332234
Baseline Brier             0.335636
dtype: float64

## 7. Model 3: rating only

One indicator column per rating grade found in training.

In [20]:
rating_train = pd.DataFrame(index=train.index)
rating_val = pd.DataFrame(index=validation.index)

for grade in train["customer_risk_rating"].unique():
    rating_train[grade] = np.where(train["customer_risk_rating"] == grade, 1, 0)
    rating_val[grade] = np.where(validation["customer_risk_rating"] == grade, 1, 0)

rating_model = LogisticRegression()
rating_fit = rating_model.fit(rating_train, train["target_default_12m"])

rating_probabilities = rating_fit.predict_proba(rating_val)[:, 1]
rating_auc = roc_auc_score(validation["target_default_12m"], rating_probabilities)
rating_brier = brier_score_loss(validation["target_default_12m"], rating_probabilities)

pd.DataFrame(
    {
        "AUC": [auc, interest_auc, rating_auc],
        "Brier": [brier, interest_brier, rating_brier],
    },
    index=["Original model", "With interest rate", "Rating only"],
)

,AUC,Brier
Original model,0.625710,0.077648
With interest rate,0.641143,0.077586
Rating only,0.682674,0.076993


### 7.1. Validation calibration

In [21]:
rating_observed, rating_predicted = calibration_curve(
    validation["target_default_12m"],
    rating_probabilities,
    n_bins=5,
    strategy="quantile",
)

pd.DataFrame(
    {
        "predicted_%": rating_predicted * 100,
        "observed_%": rating_observed * 100,
    }
)

,predicted_%,observed_%
0,2.763817,2.635298
1,6.544229,6.360041
2,9.345893,10.375881
3,11.676357,14.659007
4,15.261301,16.860907


### 7.2. Test results

In [22]:
# Match the rating columns from training.
rating_test = pd.DataFrame(index=test.index)

for grade in rating_train.columns:
    rating_test[grade] = np.where(test["customer_risk_rating"] == grade, 1, 0)

rating_test_probabilities = rating_fit.predict_proba(rating_test)[:, 1]
rating_test_auc = roc_auc_score(test["target_default_12m"], rating_test_probabilities)
rating_test_brier = brier_score_loss(
    test["target_default_12m"], rating_test_probabilities
)

display(
    pd.Series(
        {
            "Predicted default (%)": rating_test_probabilities.mean() * 100,
            "Observed default (%)": test["target_default_12m"].mean() * 100,
            "AUC": rating_test_auc,
            "Model Brier": rating_test_brier,
            "Baseline Brier": test_brier_baseline,
        }
    )
)

rating_test_observed, rating_test_predicted = calibration_curve(
    test["target_default_12m"], rating_test_probabilities, n_bins=5, strategy="quantile"
)

pd.DataFrame(
    {
        "predicted_%": rating_test_predicted * 100,
        "observed_%": rating_test_observed * 100,
    }
)

Predicted default (%)    6.955084
Observed default (%)     7.531990
AUC                      0.678827
Model Brier              0.067709
Baseline Brier           0.069739
dtype: float64

,predicted_%,observed_%
0,2.763817,2.721433
1,5.418199,5.265707
2,7.417479,7.694485
3,9.345893,9.833700
4,13.001893,15.785240


### 7.3. Denmark results

In [23]:
rating_denmark = pd.DataFrame(index=denmark_loans.index)

for grade in rating_train.columns:
    rating_denmark[grade] = np.where(
        denmark_loans["customer_risk_rating"] == grade, 1, 0
    )

rating_denmark_probabilities = rating_fit.predict_proba(rating_denmark)[:, 1]
rating_denmark_brier = brier_score_loss(denmark_target, rating_denmark_probabilities)

pd.Series(
    {
        "Loans": len(denmark_target),
        "Actual defaults": denmark_target.sum(),
        "Predicted default (%)": rating_denmark_probabilities.mean() * 100,
        "Observed default (%)": denmark_target.mean() * 100,
        "Model Brier": rating_denmark_brier,
        "Baseline Brier": (denmark_target - train["target_default_12m"].mean())
        .pow(2)
        .mean(),
    }
)

Loans                    182.000000
Actual defaults           72.000000
Predicted default (%)     15.718886
Observed default (%)      39.560440
Model Brier                0.292682
Baseline Brier             0.335636
dtype: float64

### 7.4. Compare the first three models

In [24]:
comparison = pd.DataFrame(
    {
        "Validation AUC": [auc, interest_auc, rating_auc],
        "Validation Brier": [brier, interest_brier, rating_brier],
        "Test AUC": [test_auc, interest_test_auc, rating_test_auc],
        "Test Brier": [test_brier, interest_test_brier, rating_test_brier],
        "Denmark Brier": [
            brier_score_loss(denmark_target, denmark_probabilities),
            interest_denmark_brier,
            rating_denmark_brier,
        ],
    },
    index=["Original model", "With interest rate", "Rating only"],
)

comparison.round(6)

,Validation AUC,Validation Brier,Test AUC,Test Brier,Denmark Brier
Original model,0.625710,0.077648,0.615831,0.068934,0.331602
With interest rate,0.641143,0.077586,0.635451,0.069404,0.332234
Rating only,0.682674,0.076993,0.678827,0.067709,0.292682


## 8. Model 4: add projected return

Model 2 plus projected NPV return.

In [25]:
train_pricing = train_interest.copy()
val_pricing = val_interest.copy()

npv_mean = train["projected_npv_return"].mean()
npv_std = train["projected_npv_return"].std()

train_pricing["projected_npv_return"] = (
    train["projected_npv_return"] - npv_mean
) / npv_std

val_pricing["projected_npv_return"] = (
    validation["projected_npv_return"] - npv_mean
) / npv_std

pricing_model = LogisticRegression()
pricing_fit = pricing_model.fit(train_pricing, train["target_default_12m"])

pricing_probabilities = pricing_fit.predict_proba(val_pricing)[:, 1]
pricing_auc = roc_auc_score(validation["target_default_12m"], pricing_probabilities)
pricing_brier = brier_score_loss(
    validation["target_default_12m"], pricing_probabilities
)
pd.DataFrame(
    {
        "AUC": [auc, interest_auc, rating_auc, pricing_auc],
        "Brier": [brier, interest_brier, rating_brier, pricing_brier],
    },
    index=[
        "Original model",
        "With interest rate",
        "Rating only",
        "With interest rate + projected return",
    ],
)

,AUC,Brier
Original model,0.625710,0.077648
With interest rate,0.641143,0.077586
Rating only,0.682674,0.076993
With interest rate + projected return,0.711781,0.075735


### 8.1. Validation calibration

In [26]:
pricing_observed, pricing_predicted = calibration_curve(
    validation["target_default_12m"],
    pricing_probabilities,
    n_bins=5,
    strategy="quantile",
)

pd.DataFrame(
    {
        "predicted_%": pricing_predicted * 100,
        "observed_%": pricing_observed * 100,
    }
)

,predicted_%,observed_%
0,3.130728,2.557781
1,4.556809,4.368832
2,6.023139,6.772171
3,8.053679,10.471688
4,15.710023,19.307439


### 8.2. Test results

In [27]:
test_pricing = test_interest.copy()
test_pricing["projected_npv_return"] = (
    test["projected_npv_return"] - npv_mean
) / npv_std

pricing_test_probabilities = pricing_fit.predict_proba(test_pricing)[:, 1]

pricing_test_auc = roc_auc_score(test["target_default_12m"], pricing_test_probabilities)
pricing_test_brier = brier_score_loss(
    test["target_default_12m"], pricing_test_probabilities
)

display(
    pd.Series(
        {
            "Predicted default (%)": pricing_test_probabilities.mean() * 100,
            "Observed default (%)": test["target_default_12m"].mean() * 100,
            "AUC": pricing_test_auc,
            "Model Brier": pricing_test_brier,
            "Baseline Brier": test_brier_baseline,
        }
    )
)

pricing_test_observed, pricing_test_predicted = calibration_curve(
    test["target_default_12m"],
    pricing_test_probabilities,
    n_bins=5,
    strategy="quantile",
)

pd.DataFrame(
    {
        "predicted_%": pricing_test_predicted * 100,
        "observed_%": pricing_test_observed * 100,
    }
)

Predicted default (%)    8.351646
Observed default (%)     7.531990
AUC                      0.690874
Model Brier              0.067340
Baseline Brier           0.069739
dtype: float64

,predicted_%,observed_%
0,3.185366,2.060775
1,4.914808,4.371362
2,6.882091,6.975661
3,9.539126,8.895616
4,17.236969,15.356852


### 8.3. Denmark results

In [28]:
denmark_pricing = denmark_interest.copy()
denmark_pricing["projected_npv_return"] = (
    denmark_loans["projected_npv_return"] - npv_mean
) / npv_std

pricing_denmark_probabilities = pricing_fit.predict_proba(denmark_pricing)[:, 1]

pricing_denmark_brier = brier_score_loss(denmark_target, pricing_denmark_probabilities)

pd.Series(
    {
        "Loans": len(denmark_target),
        "Actual defaults": denmark_target.sum(),
        "Predicted default (%)": pricing_denmark_probabilities.mean() * 100,
        "Observed default (%)": denmark_target.mean() * 100,
        "Model Brier": pricing_denmark_brier,
        "Baseline Brier": (denmark_target - train["target_default_12m"].mean())
        .pow(2)
        .mean(),
    }
)

Loans                    182.000000
Actual defaults           72.000000
Predicted default (%)      9.755529
Observed default (%)      39.560440
Model Brier                0.319739
Baseline Brier             0.335636
dtype: float64

## 9. Logistic model comparison

Compare scores within each dataset. Higher AUC and lower Brier are better.

In [29]:
comparison.loc["With interest rate + projected return"] = [
    pricing_auc,
    pricing_brier,
    pricing_test_auc,
    pricing_test_brier,
    pricing_denmark_brier,
]
comparison.round(6)

,Validation AUC,Validation Brier,Test AUC,Test Brier,Denmark Brier
Original model,0.625710,0.077648,0.615831,0.068934,0.331602
With interest rate,0.641143,0.077586,0.635451,0.069404,0.332234
Rating only,0.682674,0.076993,0.678827,0.067709,0.292682
With interest rate + projected return,0.711781,0.075735,0.690874,0.067340,0.319739


## 10. Gradient boosting

Use Model 4's inputs and the default settings. Early stopping is disabled because
there is a separate validation set; `random_state=42` fixes the seed.

In [30]:
from sklearn.ensemble import HistGradientBoostingClassifier

boosting_model = HistGradientBoostingClassifier(early_stopping=False, random_state=42)
boosting_model.fit(train_pricing, train["target_default_12m"])
boosting_val_probabilities = boosting_model.predict_proba(val_pricing)
boosting_val_probabilities[0:5, :]
boosting_val_default_probabilities = boosting_val_probabilities[:, 1]
boosting_val_auc = roc_auc_score(
    validation["target_default_12m"], boosting_val_default_probabilities
)
boosting_val_brier = brier_score_loss(
    validation["target_default_12m"], boosting_val_default_probabilities
)
print("Val AUC:", boosting_val_auc)
print("Val Brier:", boosting_val_brier)

Val AUC: 0.71178817719162
Val Brier: 0.07584768929546024


In [31]:
boosting_test_default_probabilities = boosting_model.predict_proba(test_pricing)[:, 1]
boosting_test_auc = roc_auc_score(
    test["target_default_12m"], boosting_test_default_probabilities
)
boosting_test_brier = brier_score_loss(
    test["target_default_12m"], boosting_test_default_probabilities
)
print("Test AUC:", boosting_test_auc)
print("Test Brier:", boosting_test_brier)

Test AUC: 0.6779859865589407
Test Brier: 0.06789892907410615


### Results

The logistic model with interest rate and projected return had the highest AUC and
lowest Brier among the four logistic models on validation and test data. Boosting
had almost the same validation AUC, but lower test AUC and higher test Brier.

## Data preparation

[prepare_data.py](prepare_data.py) keeps complete rows for the ten required columns,
using loans issued from October 2023 through March 2025 in Finland, Estonia, the
Netherlands, Latvia, and Denmark. It saves the result as `credit_data.xlsx`.

Later runs reuse this file. A newer source workbook may change the sample and results.